# 06 - Simulation and Optimization

This optional notebook shows how evaluation can guide an improvement from an initial version to a tested candidate:

```text
current version -> simulated or live sessions -> evaluation
                -> understand failures -> candidate change
                -> controlled test -> choose a version
```

You will:

1. Define a simulated user for a multi-turn conversation.
2. Find repeated patterns in evaluation results and request a suggested improvement.
3. Store prompt variants and compare them with an A/B test.
4. Clean up the module's AWS resources.

**Estimated time:** 35-60 minutes  
**Feature status:** Simulated dataset evaluation and AgentCore Insights are public preview as of August 14, 2026. Recommendations, configuration bundles, and A/B testing are optional features that may not be available in every Region or CLI version. The first cell checks whether your CLI includes the commands used here.

## 1. Check the optional commands

AgentCore features can arrive at different times across CLI versions and AWS Regions. The next cell confirms that your installed CLI recognizes each optional command before you try to use it.

This notebook was verified with AgentCore Python SDK `1.21.0` and AgentCore CLI `0.27.0` on August 18, 2026.

In [ ]:
import json

import pandas as pd
from IPython.display import Markdown, display

from src.workshop_utils import run_cli, run_cli_json

optional_commands = [
    ("run", "insights"),
    ("run", "recommendation"),
    ("add", "config-bundle"),
    ("run", "ab-test"),
]

command_rows = []
for command in optional_commands:
    help_output = run_cli(*command, "--help").stdout
    description = next(
        (
            line
            for line in help_output.splitlines()
            if line.strip() and not line.startswith("Usage:")
        ),
        "Command is available.",
    )
    command_rows.append(
        {
            "Command": f"agentcore {' '.join(command)}",
            "Available": bool(help_output.strip()),
            "Purpose": description,
        }
    )

display(pd.DataFrame(command_rows))


def show_job_summary(payload, job_type):
    row = {
        "Job type": job_type,
        "Job ID": payload.get("id"),
        "Name": payload.get("name"),
        "Status": payload.get("status", "Submitted"),
        "Created": payload.get("createdAt"),
        "Updated": payload.get("updatedAt"),
    }
    display(pd.DataFrame([row]))


def show_job_output(payload):
    for key in ("results", "result", "insights", "recommendations", "output"):
        value = payload.get(key)
        if value is None:
            continue
        display(Markdown(f"### {key.replace('_', ' ').title()}"))
        records = value if isinstance(value, list) else [value]
        if records and all(isinstance(item, dict) for item in records):
            display(pd.json_normalize(records))
        else:
            display(value)
        return
    print(
        "No detailed output was included in this response. "
        "The full job record is available in the result variable."
    )

## 2. Define a simulated actor

A simulated actor is a model that plays the role of the user. Give it enough information to produce a realistic conversation:

- a concrete goal
- relevant context
- communication traits that affect how it asks questions
- a maximum turn count
- requirements that describe a successful outcome

Simulation is most useful for conversations that are difficult to write turn by turn, such as ambiguous requests and changing goals. It does not replace human review, and generating more simulated conversations does not automatically make an evaluation more trustworthy.

In [ ]:
from bedrock_agentcore.evaluation import (
    ActorProfile,
    Dataset,
    SimulatedScenario,
)

simulated_dataset = Dataset(
    scenarios=[
        SimulatedScenario(
            scenario_id="ambiguous-portland-follow-up",
            scenario_description=(
                "A beginner asks about Portland without a state, "
                "then clarifies and requests a Seattle comparison."
            ),
            actor_profile=ActorProfile(
                traits={
                    "technical_level": "beginner",
                    "communication_style": "brief follow-up questions",
                },
                context=(
                    "The user means Portland, Oregon and is comparing "
                    "potential relocation cities."
                ),
                goal=(
                    "Obtain the workshop facts for Portland, OR and "
                    "a correct comparison with Seattle, WA."
                ),
            ),
            input="Tell me about Portland.",
            max_turns=6,
            assertions=[
                "The agent should clarify the state before looking up Portland.",
                "The final comparison should use Portland, OR and Seattle, WA.",
                "The agent should not invent facts for Portland, ME.",
            ],
        )
    ]
)
display(
    pd.DataFrame(
        [
            {
                "Scenario": "ambiguous-portland-follow-up",
                "Starting message": "Tell me about Portland.",
                "User goal": (
                    "Get Portland, OR facts and compare them with Seattle, WA."
                ),
                "Maximum turns": 6,
            }
        ]
    )
)
display(
    pd.DataFrame(
        [
            {"Actor trait": key, "Value": value}
            for key, value in simulated_dataset.scenarios[0].actor_profile.traits.items()
        ]
    )
)
print("Success requirements:")
for assertion in simulated_dataset.scenarios[0].assertions:
    print(f"- {assertion}")

## 3. Configure the simulation runner

The actor model generates the user's next message. The deployed `CityAnalyst` answers those messages and remains the system being tested. Keeping these roles separate makes it clear that a strong actor model does not improve the agent's answers; it only creates the conversation.

In [ ]:
import os

from bedrock_agentcore.evaluation import (
    CloudWatchAgentSpanCollector,
    EvaluationRunConfig,
    EvaluatorConfig,
    OnDemandEvaluationDatasetRunner,
    SimulationConfig,
)
from src.workshop_utils import RuntimeInvoker, load_runtime_info

runtime = load_runtime_info()
invoker = RuntimeInvoker(runtime)
collector = CloudWatchAgentSpanCollector(
    log_group_name=runtime.log_group_name,
    region=runtime.region,
    max_wait_seconds=180,
    poll_interval_seconds=10,
)

actor_model = os.getenv(
    "AGENTCORE_SIMULATOR_MODEL_ID",
    "global.anthropic.claude-sonnet-4-6",
)
simulation_run_config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.GoalSuccessRate",
            "Builtin.InstructionFollowing",
            "Builtin.Helpfulness",
        ]
    ),
    evaluation_delay_seconds=0,
    max_concurrent_scenarios=1,
    simulation_config=SimulationConfig(model_id=actor_model),
)

In [ ]:
RUN_SIMULATION = False

if RUN_SIMULATION:
    simulation_result = OnDemandEvaluationDatasetRunner(
        region=runtime.region
    ).run(
        config=simulation_run_config,
        dataset=simulated_dataset,
        agent_invoker=invoker,
        span_collector=collector,
    )
    simulation_rows = []
    for scenario_result in simulation_result.scenario_results:
        if scenario_result.status != "COMPLETED":
            simulation_rows.append(
                {
                    "scenario": scenario_result.scenario_id,
                    "evaluator": "Scenario execution",
                    "value": None,
                    "label": "ERROR",
                    "explanation": scenario_result.error,
                }
            )
            continue

        for evaluator_result in scenario_result.evaluator_results:
            for score in evaluator_result.results:
                error = score.get("errorMessage") or score.get("errorCode")
                simulation_rows.append(
                    {
                        "scenario": scenario_result.scenario_id,
                        "evaluator": evaluator_result.evaluator_id,
                        "value": score.get("value"),
                        "label": "ERROR" if error else score.get("label"),
                        "explanation": (
                            error or score.get("explanation")
                        ),
                    }
                )

    if not simulation_rows:
        print("The simulation completed without returning evaluator results.")
    else:
        simulation_df = pd.DataFrame(simulation_rows)
        display(simulation_df[["scenario", "evaluator", "value", "label"]])
        for row in simulation_rows:
            display(
                Markdown(
                    f"**{row['scenario']} - {row['evaluator']}**  \n"
                    f"Score: `{row['value'] if row['value'] is not None else 'Not available'}` | "
                    f"Label: `{row['label'] or 'Not available'}`\n\n"
                    f"{row['explanation'] or 'No explanation was returned.'}"
                )
            )
else:
    print(
        "Simulation invokes both an actor model and the agent. "
        "Set RUN_SIMULATION=True when you are ready."
    )

## 4. Understand a low score before making a change

Before changing the prompt, inspect:

- what the user wanted and the turn where the problem began
- tool selection and parameters
- whether the tool returned useful data
- which instruction the agent missed
- whether the evaluator misunderstood the conversation

Changing the prompt is only one possible fix. The problem may instead be an unclear tool description, missing input check, conversation-memory bug, incorrect reference information, or an evaluator that needs improvement.

## 5. Insights jobs

An Insights job reviews a group of recent sessions and looks for repeated patterns. For example, it may find that several failed conversations share the same user intent or tool error. Current CLI insight IDs include:

- `Builtin.Insight.FailureAnalysis`
- `Builtin.Insight.UserIntent`
- `Builtin.Insight.ExecutionSummary`

The guarded cell below reviews the last seven days and waits for the job to finish.

Treat each pattern as a lead to investigate, not as a final diagnosis. Read the supporting traces and ask a person to confirm the pattern before changing the agent.

In [ ]:
RUN_INSIGHTS = False

if RUN_INSIGHTS:
    insights_result = run_cli_json(
        "run",
        "insights",
        "--runtime",
        "CityAnalyst",
        "--insights",
        "Builtin.Insight.FailureAnalysis",
        "Builtin.Insight.UserIntent",
        "--evaluator",
        "Builtin.GoalSuccessRate",
        "--lookback-days",
        "7",
        "--name",
        "CityAnalystFailures",
        "--wait",
    )
    show_job_summary(insights_result, "Insights")
    show_job_output(insights_result)
else:
    print("Set RUN_INSIGHTS=True after CityAnalyst has representative sessions.")

## 6. Recommendation jobs

A Recommendation job uses traces and evaluator results to suggest a revised system prompt or tool description. The guarded cell submits the current CityAnalyst system prompt and waits for a proposed version.

The suggestion is a candidate, not an automatic upgrade. Keep the current prompt, review the proposed text, rerun the same dataset, and compare individual failures before choosing whether to use it.

In [ ]:
RUN_RECOMMENDATION = False

if RUN_RECOMMENDATION:
    recommendation_result = run_cli_json(
        "run",
        "recommendation",
        "--type",
        "system-prompt",
        "--runtime",
        "CityAnalyst",
        "--evaluator",
        "Builtin.GoalSuccessRate",
        "--prompt-file",
        "app/CityAnalyst/system_prompt.txt",
        "--lookback",
        "7",
        "--run",
        "CityPromptRecommendation",
        "--wait",
    )
    show_job_summary(recommendation_result, "Recommendation")
    show_job_output(recommendation_result)
else:
    print("Set RUN_RECOMMENDATION=True when recent sessions are ready for analysis.")

## 7. Configuration bundles

A configuration bundle stores settings such as a system prompt as a named, versioned resource. This lets you compare prompt versions without creating a separate copy of the application code for each one.

Example component file:

In [ ]:
import json
from pathlib import Path

control_components = {
    "{{runtime:CityAnalyst}}": {
        "configuration": {
            "systemPrompt": Path(
                "app/CityAnalyst/system_prompt.txt"
            ).read_text()
        }
    }
}
Path("generated").mkdir(exist_ok=True)
Path("generated/control_bundle.json").write_text(
    json.dumps(control_components, indent=2) + "\n"
)
display(
    pd.DataFrame(
        [
            {
                "Bundle role": "Control (current version)",
                "Runtime": "CityAnalyst",
                "Setting": "System prompt",
                "Characters": len(
                    control_components["{{runtime:CityAnalyst}}"]
                    ["configuration"]["systemPrompt"]
                ),
                "Generated file": "generated/control_bundle.json",
            }
        ]
    )
)

The next cell can add the bundle to the local project, deploy it to AWS, and list the versions stored by the service. Each action has a separate flag so you can review one step before starting the next.

The version list tells you exactly which prompt was used in an experiment. The `agentcore config-bundle diff` command can then show how two versions differ.

In [ ]:
from src.workshop_utils import MODULE_ROOT

CONFIG_BUNDLE_NAME = "CityPromptControl"
CREATE_CONFIG_BUNDLE = False
DEPLOY_CONFIG_BUNDLE = False
LIST_CONFIG_BUNDLE_VERSIONS = False

project_config = json.loads(
    (MODULE_ROOT / "agentcore" / "agentcore.json").read_text()
)
configured_bundles = {
    item["name"] for item in project_config.get("configBundles", [])
}

if CREATE_CONFIG_BUNDLE and CONFIG_BUNDLE_NAME not in configured_bundles:
    result = run_cli(
        "add",
        "config-bundle",
        "--name",
        CONFIG_BUNDLE_NAME,
        "--description",
        "Baseline CityAnalyst prompt",
        "--components-file",
        "generated/control_bundle.json",
        "--branch",
        "main",
        "--commit-message",
        "Workshop baseline",
    )
    print(result.stdout.strip())
elif CREATE_CONFIG_BUNDLE:
    print(f"{CONFIG_BUNDLE_NAME} is already present in agentcore.json.")
else:
    print("Set CREATE_CONFIG_BUNDLE=True to add the control bundle locally.")

if DEPLOY_CONFIG_BUNDLE:
    bundle_deploy_result = run_cli_json(
        "deploy", "--target", "default", "--yes"
    )
    print(f"Deployed {CONFIG_BUNDLE_NAME} to the default target.")

if LIST_CONFIG_BUNDLE_VERSIONS:
    bundle_versions_result = run_cli_json(
        "config-bundle",
        "versions",
        "--name",
        CONFIG_BUNDLE_NAME,
    )
    bundle_versions = bundle_versions_result.get("versions", [])
    if bundle_versions:
        display(pd.json_normalize(bundle_versions))
    else:
        print("No deployed bundle versions were returned.")

## 8. Gateway-backed A/B tests

An A/B test sends part of the traffic to a **control** version and the rest to a **treatment** version. The control is the current prompt or configuration; the treatment is the candidate you want to test. AgentCore uses a deployed Gateway to route that traffic, then online evaluation scores both groups.

Prerequisites:

- a deployed Gateway
- a control and treatment configuration, represented by bundle versions or Gateway targets
- online evaluation configured for the compared traffic
- enough representative traffic to compare the groups fairly

Provide the deployed Gateway and treatment bundle names in the cell below before enabling the test.

Choose the treatment only after reviewing the size of the improvement, evaluator reliability, results for important user groups, latency and error metrics, and your ability to return to the control version.

In [ ]:
RUN_AB_TEST = False
AB_TEST_GATEWAY = ""
TREATMENT_BUNDLE_NAME = ""

if RUN_AB_TEST:
    if not AB_TEST_GATEWAY or not TREATMENT_BUNDLE_NAME:
        raise ValueError(
            "Set AB_TEST_GATEWAY and TREATMENT_BUNDLE_NAME before running the test."
        )
    ab_test_result = run_cli_json(
        "run",
        "ab-test",
        "--name",
        "CityPromptExperiment",
        "--gateway",
        AB_TEST_GATEWAY,
        "--runtime",
        "CityAnalyst",
        "--control-bundle",
        "CityPromptControl",
        "--control-version",
        "LATEST",
        "--treatment-bundle",
        TREATMENT_BUNDLE_NAME,
        "--treatment-version",
        "LATEST",
        "--online-eval",
        "CityQualityMonitor",
        "--control-weight",
        "50",
        "--treatment-weight",
        "50",
    )
    show_job_summary(ab_test_result, "A/B test")
    show_job_output(ab_test_result)
else:
    print("Set RUN_AB_TEST=True after all listed prerequisites are deployed.")

## 9. The complete improvement loop

| Stage | Evidence |
|---|---|
| Current version | Record the agent, dataset, evaluator versions, and score distribution |
| Understand failures | Read traces and explanations, add human review, and optionally run Insights |
| Make one change | Update the prompt, tool description, input checks, model, or code |
| Test without live traffic | Run the same dataset with the same evaluator versions |
| Test with controlled traffic | Use online sampling or a Gateway A/B test |
| Choose a version | Record the decision and keep a clear rollback point |

Do not choose a version from the average score alone. Check which kinds of failures improved and whether any previously successful behavior became worse.

## 10. Cleanup

The cleanup helper asks CloudFormation to remove the AWS resources created by this module, then restores your local AgentCore project file.

It also removes generated notebook files while keeping the `generated/` directory itself. It does not delete Lambda functions managed elsewhere, customer-managed KMS keys, or log groups that are configured to be retained.

In [ ]:
import subprocess
import sys
from src.workshop_utils import MODULE_ROOT

RUN_CLEANUP = False

if RUN_CLEANUP:
    subprocess.run(
        [sys.executable, "src/cleanup.py", "--target", "default", "--yes"],
        cwd=MODULE_ROOT,
        check=True,
    )
else:
    print(
        "Set RUN_CLEANUP=True after completing the workshop, "
        "then verify retained resources in the AWS console."
    )

## Module complete

You can now use AgentCore evaluation from a first test through ongoing improvement:

1. run one agent
2. inspect conversations, traces, and individual tool calls
3. evaluate recent behavior with built-ins
4. add clear reference information and reusable datasets
5. create focused custom evaluators and compare them with human labels
6. operate batch and online monitoring
7. simulate harder conversations and compare candidate improvements

The core habit is simple: describe the behavior you care about, collect the evidence needed to judge it, choose a focused evaluator, and test that evaluator before it controls an automated decision.